# Generalization Test: Feature-Free dVRK Dataset, PSM1

This notebook tests the force-estimation pipeline on a second dataset:

```text
D:\_RESEARCH\_DATA\dvrk_force_estimation_feature_free\Data
```

For now we focus on **PSM1** only. The first baseline is now computed directly from the exported torque prediction path:

$$
\hat{	au}_{\mathrm{ext}} = 	au_{\mathrm{meas}} - 	au_{\mathrm{LSTM}},
\qquad
\hat F = J^{-T}\hat{	au}_{\mathrm{ext}}.
$$

Then the PSM1 force-frame rotation from `LSTM_analyze_PSM1.ipynb` is applied. The force sensor is used only for validation and RMSE calculation.

In [ ]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from IPython.display import Image, display



## Paths

The split folders are:

- `train_csv`
- `val_csv`
- `test_csv`

The current exported LSTM prediction files appear to be for PSM1, so this notebook uses them directly.


In [ ]:
DATA_ROOT = Path(r"D:\_RESEARCH\_DATA\dvrk_force_estimation_feature_free\Data")
TEST_CSV = DATA_ROOT / "test_csv"

JOINT_CSV = TEST_CSV / "joints" / "interpolated_all_joints.csv"
JACOBIAN_CSV = TEST_CSV / "jacobian" / "interpolated_all_jacobian.csv"
PREDICTED_TORQUE_CSV = TEST_CSV / "lstm_seal_pred_filtered_torque_colon_9_26.csv"
SENSOR_CSV = TEST_CSV / "sensor" / "interpolated_all_sensor.csv"
REFERENCE_COMPARISON_PNG = TEST_CSV / "lstm_force_comparison.png"

OUTPUT_DIR = Path.cwd() / "generalization_psm1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [DATA_ROOT, TEST_CSV, JOINT_CSV, JACOBIAN_CSV, PREDICTED_TORQUE_CSV, SENSOR_CSV]:
    print(path, "OK" if path.exists() else "MISSING")

## Coordinate rotation used by the reference notebook

The reference notebook rotates the robot-frame force into the force-sensor plotting frame using:

$$
F_{sensor} = A F_{robot},
$$

where `A` is built from the `flip_y_axis`, `basis`, `Rz(angle_z)`, `Rx(-45 deg)`, and `axis_correction` matrices below. The default reference force-estimation cell uses:

$$
	heta_z = -22^\circ.
$$

This rotation is applied after the direct torque-to-force computation.

In [ ]:

def reference_force_rotation(angle_z_deg=-22.0):
    flip_y_axis = np.diag([-1.0, -1.0, 1.0])

    basis = np.array([
        [0, 0, -1],
        [0, 1, 0],
        [1, 0, 0],
    ], dtype=float)

    angle_z = np.deg2rad(angle_z_deg)
    Rz_x = np.array([
        [np.cos(angle_z), -np.sin(angle_z), 0.0],
        [np.sin(angle_z),  np.cos(angle_z), 0.0],
        [0.0,              0.0,             1.0],
    ], dtype=float)

    Rx_minus_45 = np.array([
        [1.0,  0.0,     0.0],
        [0.0,  0.7071,  0.7071],
        [0.0, -0.7071,  0.7071],
    ], dtype=float)

    rotated = Rz_x @ Rx_minus_45 @ flip_y_axis
    T_transpose = np.linalg.inv(rotated) @ basis

    axis_correction = np.array([
        [0.0,  1.0, 0.0],
        [-1.0, 0.0, 0.0],
        [0.0,  0.0, 1.0],
    ], dtype=float)

    return axis_correction, T_transpose

axis_correction, T_transpose = reference_force_rotation(angle_z_deg=-22.0)

def apply_coord_transform(force_np):
    force_sensor = force_np @ T_transpose
    return (axis_correction @ force_sensor.T).T

print("axis_correction =")
print(axis_correction)
print("T_transpose =")
print(T_transpose)


## Direct torque-residual force-estimation baseline

This is the baseline before adding observer/Kalman/CI methods. The direct force output is smoothed with a 50-sample moving average, matching the reference `LSTM_analyze_PSM1.ipynb` plotting style.

Inputs:

- `joints/interpolated_all_joints.csv`: measured joint states and measured joint torques.
- `jacobian/interpolated_all_jacobian.csv`: Jacobian at each timestamp.
- `lstm_seal_pred_filtered_torque_colon_9_26.csv`: LSTM-predicted free-space joint torque.
- `sensor/interpolated_all_sensor.csv`: measured force sensor validation signal.

The computation below does not use `lstm_force.csv`. It directly computes:

$$
\hat{	au}_{\mathrm{ext},k}=	au_{\mathrm{meas},k}-	au_{\mathrm{LSTM},k},
\qquad
\hat F_k=J_k^{-T}\hat{	au}_{\mathrm{ext},k}.
$$

In [ ]:
DIRECT_BASELINE_MOVING_AVERAGE_WINDOW = 50


def trailing_moving_average(data, window_size):
    if window_size <= 1:
        return data.copy()
    kernel = np.ones(window_size, dtype=float) / float(window_size)
    return np.apply_along_axis(lambda col: np.convolve(col, kernel, mode="same"), 0, data)


def build_jacobian_from_csv(jacobian_data):
    return jacobian_data[:, 1:37].reshape(-1, 6, 6)


def compute_direct_force_from_torque(joint_csv=JOINT_CSV, jacobian_csv=JACOBIAN_CSV, torque_csv=PREDICTED_TORQUE_CSV, sensor_csv=SENSOR_CSV):
    joint_data = np.loadtxt(joint_csv, delimiter=",")
    jacobian_data = np.loadtxt(jacobian_csv, delimiter=",")
    torque_data = np.loadtxt(torque_csv, delimiter=",")
    sensor_data = np.loadtxt(sensor_csv, delimiter=",")

    n_samples = min(joint_data.shape[0], jacobian_data.shape[0], torque_data.shape[0])
    joint_data = joint_data[:n_samples]
    jacobian_data = jacobian_data[:n_samples]
    torque_data = torque_data[:n_samples]

    time = joint_data[:, 0]
    measured_torque = joint_data[:, 13:19]
    predicted_free_space_torque = torque_data[:, 1:7]
    external_torque = measured_torque - predicted_free_space_torque

    jacobian = build_jacobian_from_csv(jacobian_data)
    raw_force = np.empty((n_samples, 6), dtype=float)
    for idx in range(n_samples):
        raw_force[idx] = np.linalg.solve(jacobian[idx].T, external_torque[idx])
    raw_force[:, :3] = apply_coord_transform(raw_force[:, :3])
    force = trailing_moving_average(raw_force, DIRECT_BASELINE_MOVING_AVERAGE_WINDOW)

    sensor_time = sensor_data[:, 0]
    sensor_force = sensor_data[:, 1:4]
    sensor_interp = np.empty((n_samples, 3), dtype=float)
    for axis in range(3):
        interpolator = interp1d(sensor_time, sensor_force[:, axis], bounds_error=False, fill_value="extrapolate")
        sensor_interp[:, axis] = interpolator(time)

    comparison = np.column_stack([time, force[:, :3], sensor_interp])
    rmse = np.sqrt(np.nanmean((force[:, :3] - sensor_interp) ** 2, axis=0))
    return comparison, rmse, force, raw_force, external_torque, joint_data, jacobian_data, torque_data


comparison, rmse, direct_force_full, direct_raw_force_full, direct_external_torque, direct_joint_data, direct_jacobian_data, direct_torque_data = compute_direct_force_from_torque()
computed_csv = OUTPUT_DIR / "direct_force_comparison_computed.csv"
np.savetxt(
    computed_csv,
    comparison,
    delimiter=",",
    header="time,predicted_Fx,predicted_Fy,predicted_Fz,measured_Fx,measured_Fy,measured_Fz",
    comments="",
)

print(f"Direct torque-residual force-estimation RMSE (N), moving average window = {DIRECT_BASELINE_MOVING_AVERAGE_WINDOW}:")
for label, value in zip(["Fx", "Fy", "Fz"], rmse):
    print(f"  {label}: {value:.4f}")
print(f"  mean: {rmse.mean():.4f}")
print("Saved:", computed_csv)

In [ ]:

def plot_regular_force_comparison(comparison, rmse, fig_path):
    labels = ["Fx", "Fy", "Fz"]
    time = comparison[:, 0]
    pred_force = comparison[:, 1:4]
    measured_force = comparison[:, 4:7]

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), constrained_layout=True)
    fig.suptitle("Direct Torque-Residual Force Estimation")

    for axis, ax in enumerate(axes):
        ax.plot(time, measured_force[:, axis], "b", label="measured")
        ax.plot(time, pred_force[:, axis], "r", label="predicted")
        ax.set_title(f"{labels[axis]}, RMSE = {rmse[axis]:.4f}")
        ax.set_ylabel("Force / N")
        ax.grid(True)

    axes[-1].set_xlabel("Time / s")
    axes[0].legend(loc="upper right")
    fig.savefig(fig_path, dpi=150)
    plt.show()
    return fig_path

reproduced_png = OUTPUT_DIR / "direct_force_comparison_computed.png"
plot_regular_force_comparison(comparison, rmse, reproduced_png)
print("Saved:", reproduced_png)


## Direct-compute output

The plot below is generated from the direct torque-residual computation, not from the saved `lstm_force.csv` file. The old `test_csv/lstm_force_comparison.png` can still be displayed as a reference image, but it is not used as an input.

In [ ]:
print("Direct-compute comparison shape:", comparison.shape)
print("Direct-compute RMSE:", rmse)
print("Mean RMSE:", rmse.mean())

In [ ]:
print("Direct-compute plot from this notebook:")
display(Image(filename=str(reproduced_png)))

if REFERENCE_COMPARISON_PNG.exists():
    print("Old saved lstm_force comparison image, shown only for visual reference:")
    display(Image(filename=str(REFERENCE_COMPARISON_PNG)))

## Baseline conclusion

The initial baseline is now the true direct torque-residual force computation. It intentionally does not read `lstm_force.csv`.

If this direct result is worse than the old saved `lstm_force.csv` image, that is useful information: it means the exported torque-residual path, indexing/windowing, Jacobian convention, or post-processing is not equivalent to the force file generated by the original LSTM analysis notebook.

The observer/Kalman/CI section below starts from this same direct torque residual path, so the comparison is internally consistent.

## Observer, Kalman, and Leak-Free CI Generalization

This section applies the same torque-residual observer/Kalman/CI concepts from the dVRK-Si contact dataset to this PSM1 feature-free test split.

Important validation rule: the force sensor is still used only for plotting and RMSE. The observer, Kalman filter, and confidence interval are computed only from robot signals, LSTM torque predictions, Jacobian, and velocity.

In [ ]:
from dataclasses import dataclass
from statistics import NormalDist

JOINT_CSV = TEST_CSV / "joints" / "interpolated_all_joints.csv"
JACOBIAN_CSV = TEST_CSV / "jacobian" / "interpolated_all_jacobian.csv"
PREDICTED_TORQUE_CSV = TEST_CSV / "lstm_seal_pred_filtered_torque_colon_9_26.csv"

CHANNELS = ("Fx", "Fy", "Fz", "Taux", "Tauy", "Tauz")
JOINT_VELOCITY_CHANNELS = ("q1_dot", "q2_dot", "q3_dot", "q4_dot", "q5_dot", "q6_dot")
CARTESIAN_VELOCITY_CHANNELS = ("vx", "vy", "vz", "wx", "wy", "wz")

# Keep the same hyperparameters used in the previous notebook/script.
OBSERVER_GAIN = 15.0
BIAS_GAIN = 0.0
DEADBAND_MULTIPLIER = 20.0
JOINT_DEADBAND_MULTIPLIERS = (1.0, 1.0, 0.25, 1.0, 1.0, 1.0)
SMOOTH_WIDTH_MULTIPLIER = 0.8
KALMAN_PROCESS_SCALE = 0.08
KALMAN_MEASUREMENT_SCALE = 1.0
CI_BINS = 12
CONFIDENCE = 0.95
LOW_FORCE_GAIN = 3.0
FORCE_SCALE = 10.0
OBSERVER_DISAGREEMENT_GAIN = 0.8
RELATIVE_FORCE_UNCERTAINTY = 0.10
FZ_UNCERTAINTY_GAIN = 2.0
MAX_CI_HALF_WIDTH = 12.0

for path in [JOINT_CSV, JACOBIAN_CSV, PREDICTED_TORQUE_CSV]:
    print(path, "OK" if path.exists() else "MISSING")

### Equations

The torque residual is the same signal used before:

$$
\hat{\tau}_{\mathrm{ext},k}=\tau_{\mathrm{meas},k}-\tau_{\mathrm{LSTM},k}
$$

The observer variant is:

$$
\dot r = K_i\left(\hat{\tau}_{\mathrm{ext}} - r\right)
$$

The Kalman filter variant uses a random-walk external torque state:

$$
\tau_{\mathrm{ext},k}=\tau_{\mathrm{ext},k-1}+w_k
$$

$$
y_k=\tau_{\mathrm{meas},k}-\tau_{\mathrm{LSTM},k}=\tau_{\mathrm{ext},k}+v_k
$$

Then force is projected through the Jacobian:

$$
\hat F_k = J_k^{-T}\hat\tau_{\mathrm{ext},k}
$$

and the PSM1 force-coordinate rotation from `LSTM_analyze_PSM1.ipynb` is applied.

In [ ]:
@dataclass
class GeneralizationResults:
    time: np.ndarray
    external_torque: np.ndarray
    raw_force: np.ndarray
    force: np.ndarray
    real_force: np.ndarray
    ci_lower: np.ndarray
    ci_upper: np.ndarray
    ci_half_width: np.ndarray
    rmse: np.ndarray
    sigma_bin_centers: np.ndarray
    sigma_by_channel: np.ndarray
    joint_velocity: np.ndarray
    cartesian_velocity: np.ndarray
    cartesian_speed: np.ndarray
    estimator_mode: str


def robust_sigma(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.full(6, np.nan)
    median = np.nanmedian(values, axis=0)
    mad = np.nanmedian(np.abs(values - median), axis=0)
    sigma = 1.4826 * mad
    rms = np.sqrt(np.nanmean(values ** 2, axis=0))
    return np.maximum(sigma, 0.35 * rms)


def available_signal_sigma(torque_error):
    delta = np.diff(torque_error, axis=1)
    sigma = robust_sigma(delta.T) / np.sqrt(2.0)
    floor = 1e-6 * np.maximum(1.0, np.nanmedian(np.abs(torque_error), axis=1))
    return np.maximum(sigma, floor)


def residual_observer(torque_error, time, observer_gain=OBSERVER_GAIN):
    gain = np.asarray(observer_gain, dtype=float)
    if gain.ndim == 0:
        gain = np.full(torque_error.shape[0], float(gain))
    observed = np.empty_like(torque_error)
    observed[:, 0] = torque_error[:, 0]
    dt = np.maximum(np.diff(time[: torque_error.shape[1]]), 0.0)
    for idx, sample_dt in enumerate(dt, start=1):
        alpha = np.exp(-gain * sample_dt)
        observed[:, idx] = alpha * observed[:, idx - 1] + (1.0 - alpha) * torque_error[:, idx]
    return observed


def kalman_torque_filter(
    torque_error,
    time,
    process_scale=KALMAN_PROCESS_SCALE,
    measurement_scale=KALMAN_MEASUREMENT_SCALE,
    initial_variance_scale=25.0,
):
    measurement_sigma = measurement_scale * available_signal_sigma(torque_error)
    measurement_variance = np.maximum(measurement_sigma ** 2, 1e-18)
    process_variance = (process_scale * measurement_sigma) ** 2

    filtered = np.empty_like(torque_error)
    state = torque_error[:, 0].copy()
    covariance = initial_variance_scale * measurement_variance
    filtered[:, 0] = state

    dt = np.maximum(np.diff(time[: torque_error.shape[1]]), 0.0)
    median_dt = max(float(np.nanmedian(dt)), 1e-9) if dt.size else 1.0
    for idx, sample_dt in enumerate(dt, start=1):
        covariance = covariance + process_variance * max(sample_dt / median_dt, 1e-9)
        gain = covariance / (covariance + measurement_variance)
        state = state + gain * (torque_error[:, idx] - state)
        covariance = (1.0 - gain) * covariance
        filtered[:, idx] = state
    return filtered


def soft_deadband(values, deadband):
    return np.sign(values) * np.maximum(np.abs(values) - deadband[:, None], 0.0)


def smooth_deadband(values, deadband, width):
    width = np.maximum(width, 1e-12)
    gate = 1.0 / (1.0 + np.exp(-(np.abs(values) - deadband[:, None]) / width[:, None]))
    return values * gate


def torque_smooth_deadband_observer(torque_error, time):
    joint_scale = np.asarray(JOINT_DEADBAND_MULTIPLIERS, dtype=float)
    base_deadband = DEADBAND_MULTIPLIER * joint_scale * available_signal_sigma(torque_error)
    corrected = smooth_deadband(torque_error, base_deadband, SMOOTH_WIDTH_MULTIPLIER * base_deadband)
    return residual_observer(corrected, time, OBSERVER_GAIN)


def torque_smooth_deadband_kalman(torque_error, time):
    joint_scale = np.asarray(JOINT_DEADBAND_MULTIPLIERS, dtype=float)
    base_deadband = DEADBAND_MULTIPLIER * joint_scale * available_signal_sigma(torque_error)
    corrected = smooth_deadband(torque_error, base_deadband, SMOOTH_WIDTH_MULTIPLIER * base_deadband)
    return kalman_torque_filter(corrected, time)


def bias_gated_observer(torque_error, time):
    deadband = DEADBAND_MULTIPLIER * available_signal_sigma(torque_error)
    bias = torque_error[:, 0].copy()
    corrected = np.empty_like(torque_error)
    corrected[:, 0] = soft_deadband(torque_error[:, [0]] - bias[:, None], deadband)[:, 0]
    dt = np.maximum(np.diff(time[: torque_error.shape[1]]), 0.0)
    for idx, sample_dt in enumerate(dt, start=1):
        innovation = torque_error[:, idx] - bias
        free_space_like = np.abs(innovation) <= deadband
        bias_alpha = np.exp(-BIAS_GAIN * sample_dt)
        bias[free_space_like] = (
            bias_alpha * bias[free_space_like]
            + (1.0 - bias_alpha) * torque_error[free_space_like, idx]
        )
        corrected[:, idx] = soft_deadband(torque_error[:, [idx]] - bias[:, None], deadband)[:, 0]
    return residual_observer(corrected, time, OBSERVER_GAIN)


def two_time_scale_observer(torque_error, time):
    fast = residual_observer(torque_error, time, OBSERVER_GAIN)
    deadband = DEADBAND_MULTIPLIER * available_signal_sigma(torque_error)
    slow_bias = np.empty_like(torque_error)
    slow_bias[:, 0] = torque_error[:, 0]
    dt = np.maximum(np.diff(time[: torque_error.shape[1]]), 0.0)
    for idx, sample_dt in enumerate(dt, start=1):
        innovation = fast[:, idx] - slow_bias[:, idx - 1]
        free_space_like = np.abs(innovation) <= deadband
        alpha = np.exp(-BIAS_GAIN * sample_dt)
        slow_bias[:, idx] = slow_bias[:, idx - 1]
        slow_bias[free_space_like, idx] = (
            alpha * slow_bias[free_space_like, idx - 1]
            + (1.0 - alpha) * fast[free_space_like, idx]
        )
    return soft_deadband(fast - slow_bias, deadband)


def select_external_torque(torque_error, time, estimator_mode):
    if estimator_mode == "direct":
        return torque_error
    if estimator_mode == "observer":
        return residual_observer(torque_error, time, OBSERVER_GAIN)
    if estimator_mode == "kalman":
        return kalman_torque_filter(torque_error, time)
    if estimator_mode == "torque_smooth_deadband":
        return torque_smooth_deadband_observer(torque_error, time)
    if estimator_mode == "kalman_smooth_deadband":
        return torque_smooth_deadband_kalman(torque_error, time)
    if estimator_mode == "bias_gate":
        return bias_gated_observer(torque_error, time)
    if estimator_mode == "two_time_scale":
        return two_time_scale_observer(torque_error, time)
    raise ValueError(f"Unknown estimator_mode: {estimator_mode}")

In [ ]:


def apply_coord_transform_if_needed(force_np):
    """Apply the PSM1 force-frame rotation, even if earlier notebook cells were not rerun."""
    global axis_correction, T_transpose
    if "axis_correction" not in globals() or "T_transpose" not in globals():
        axis_correction, T_transpose = reference_force_rotation(angle_z_deg=-22.0)
    force_sensor = force_np @ T_transpose
    return (axis_correction @ force_sensor.T).T


def load_torque_pipeline_inputs(n_samples):
    joint_data = np.loadtxt(JOINT_CSV, delimiter=",")[:n_samples]
    jacobian_data = np.loadtxt(JACOBIAN_CSV, delimiter=",")[:n_samples]
    torque_data = np.loadtxt(PREDICTED_TORQUE_CSV, delimiter=",")[:n_samples]
    return joint_data, jacobian_data, torque_data


def build_jacobian_feature_free(jacobian_data):
    return jacobian_data[:, 1:37].reshape(-1, 6, 6)


def torque_error_from_inputs(joint_data, torque_data):
    n_samples = min(joint_data.shape[0], torque_data.shape[0])
    measured_torque = joint_data[:n_samples, 13:19].T
    predicted_torque = torque_data[:n_samples, 1:7].T
    return measured_torque - predicted_torque


def estimate_joint_velocity(joint_data):
    if joint_data.shape[1] >= 13:
        velocity = joint_data[:, 7:13]
        if np.nanmax(np.abs(velocity)) > 0.0:
            return velocity
    return np.gradient(joint_data[:, 1:7], joint_data[:, 0], axis=0)


def estimate_cartesian_velocity(joint_data, jacobian_data):
    qdot = estimate_joint_velocity(joint_data)
    jacobian = build_jacobian_feature_free(jacobian_data)
    cartesian_velocity = np.empty_like(qdot)
    for idx in range(qdot.shape[0]):
        cartesian_velocity[idx] = jacobian[idx] @ qdot[idx]
        cartesian_velocity[idx, :3] = apply_coord_transform_if_needed(cartesian_velocity[idx:idx + 1, :3])[0]
    cartesian_speed = np.linalg.norm(cartesian_velocity[:, :3], axis=1)
    return qdot, cartesian_velocity, cartesian_speed


def project_torque_to_force(external_torque, jacobian_data):
    jacobian = build_jacobian_feature_free(jacobian_data)
    n_samples = external_torque.shape[1]
    force = np.empty((n_samples, 6), dtype=float)
    for idx in range(n_samples):
        force[idx] = np.linalg.solve(jacobian[idx].T, external_torque[:, idx])
    force[:, :3] = apply_coord_transform_if_needed(force[:, :3])
    return force


def normal_z(confidence):
    return NormalDist().inv_cdf(0.5 + confidence / 2.0)


def leak_free_contact_dependent_ci(estimate, external_torque, joint_data, jacobian_data, torque_data):
    n_samples = estimate.shape[0]
    jacobian = build_jacobian_feature_free(jacobian_data[:n_samples])
    raw_torque_error_cols = torque_error_from_inputs(joint_data, torque_data)[:, :n_samples]
    base_tau_sigma = available_signal_sigma(raw_torque_error_cols)
    raw_torque_error_rows = raw_torque_error_cols.T
    force_magnitude = np.linalg.norm(estimate[:, :3], axis=1)
    low_force_factor = 1.0 + LOW_FORCE_GAIN * np.exp(-force_magnitude / FORCE_SCALE)

    rotation_6d = np.eye(6)
    rotation_6d[:3, :3] = axis_correction @ T_transpose.T

    sigma = np.empty_like(estimate)
    for idx in range(n_samples):
        transform = rotation_6d @ np.linalg.inv(jacobian[idx].T)
        channel_variance = (transform ** 2) @ ((base_tau_sigma * low_force_factor[idx]) ** 2)
        direct_force = transform @ raw_torque_error_rows[idx]
        propagated_sigma = np.sqrt(np.maximum(channel_variance, 0.0))
        disagreement_sigma = np.abs(direct_force - estimate[idx])
        sigma[idx] = np.sqrt(
            propagated_sigma ** 2
            + (OBSERVER_DISAGREEMENT_GAIN * disagreement_sigma) ** 2
        )

    z = 1.959963984540054 if abs(CONFIDENCE - 0.95) < 1e-12 else normal_z(CONFIDENCE)
    half_width = z * sigma
    half_width[:, :3] = np.sqrt(
        half_width[:, :3] ** 2
        + (RELATIVE_FORCE_UNCERTAINTY * np.abs(estimate[:, :3])) ** 2
    )
    half_width[:, 2] *= FZ_UNCERTAINTY_GAIN
    if MAX_CI_HALF_WIDTH is not None:
        half_width = np.minimum(half_width, MAX_CI_HALF_WIDTH)

    edges = np.quantile(force_magnitude, np.linspace(0.0, 1.0, CI_BINS + 1))
    edges = np.unique(edges)
    if edges.size < 2:
        edges = np.array([0.0, float(np.nanmax(force_magnitude)) + 1e-12])
    centers = 0.5 * (edges[:-1] + edges[1:])
    binned_sigma = np.full((centers.size, estimate.shape[1]), np.nan)
    for idx in range(centers.size):
        mask = (force_magnitude >= edges[idx]) & (force_magnitude < edges[idx + 1])
        if idx == centers.size - 1:
            mask = (force_magnitude >= edges[idx]) & (force_magnitude <= edges[idx + 1])
        if np.count_nonzero(mask) > 0:
            binned_sigma[idx] = np.nanmedian(half_width[mask] / z, axis=0)
    for col in range(estimate.shape[1]):
        valid = np.isfinite(binned_sigma[:, col])
        if np.count_nonzero(valid) == 0:
            binned_sigma[:, col] = np.nanmedian(half_width[:, col] / z)
        else:
            binned_sigma[:, col] = np.interp(centers, centers[valid], binned_sigma[valid, col])

    return estimate - half_width, estimate + half_width, half_width, centers, binned_sigma

In [ ]:
def run_torque_residual_variant(estimator_mode):
    # Use the same time span as the direct-compute baseline so all plots share one x-axis.
    n_samples = comparison.shape[0]
    joint_data, jacobian_data, torque_data = load_torque_pipeline_inputs(n_samples)
    time = comparison[:, 0]
    real_force = comparison[:, 4:7]

    torque_error = torque_error_from_inputs(joint_data, torque_data)
    external_torque = select_external_torque(torque_error, time, estimator_mode)
    raw_force = project_torque_to_force(external_torque, jacobian_data)
    force = raw_force.copy()
    rmse = np.sqrt(np.nanmean((force[:, :3] - real_force) ** 2, axis=0))

    ci_lower, ci_upper, ci_half_width, centers, sigma = leak_free_contact_dependent_ci(
        force,
        external_torque.T,
        joint_data,
        jacobian_data,
        torque_data,
    )
    joint_velocity, cartesian_velocity, cartesian_speed = estimate_cartesian_velocity(joint_data, jacobian_data)

    return GeneralizationResults(
        time=time,
        external_torque=external_torque.T,
        raw_force=raw_force,
        force=force,
        real_force=real_force,
        ci_lower=ci_lower,
        ci_upper=ci_upper,
        ci_half_width=ci_half_width,
        rmse=rmse,
        sigma_bin_centers=centers,
        sigma_by_channel=sigma,
        joint_velocity=joint_velocity,
        cartesian_velocity=cartesian_velocity,
        cartesian_speed=cartesian_speed,
        estimator_mode=estimator_mode,
    )


VARIANT_CONFIGS = {
    "direct": {"label": "Direct torque residual"},
    "observer": {"label": f"Observer, K_i={OBSERVER_GAIN:g}"},
    "kalman": {"label": "Kalman torque residual"},
    "torque_smooth_deadband": {"label": f"Smooth torque deadband + observer, db={DEADBAND_MULTIPLIER:g}"},
    "kalman_smooth_deadband": {"label": f"Smooth torque deadband + Kalman, db={DEADBAND_MULTIPLIER:g}"},
    "bias_gate": {"label": f"Bias gate, K_i={OBSERVER_GAIN:g}, db={DEADBAND_MULTIPLIER:g}"},
    "two_time_scale": {"label": f"Two-time-scale, K_i={OBSERVER_GAIN:g}, db={DEADBAND_MULTIPLIER:g}"},
}

ACTIVE_VARIANT = "kalman_smooth_deadband"

variant_results = {}
for mode in VARIANT_CONFIGS:
    result = run_torque_residual_variant(mode)
    variant_results[mode] = result
    print(f"{mode:24s} RMSE: Fx={result.rmse[0]:.4f}, Fy={result.rmse[1]:.4f}, Fz={result.rmse[2]:.4f}, mean={result.rmse.mean():.4f}")

active_results = variant_results[ACTIVE_VARIANT]
print("\nActive variant:", ACTIVE_VARIANT, "-", VARIANT_CONFIGS[ACTIVE_VARIANT]["label"])

In [ ]:
def save_generalization_results_csv(results, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    header = ["time"]
    arrays = [results.time[:, None]]
    for name, arr in (
        ("external_torque", results.external_torque),
        ("truth", np.column_stack([results.real_force, np.full((results.real_force.shape[0], 3), np.nan)])),
        ("estimate", results.force),
        ("ci_lower", results.ci_lower),
        ("ci_upper", results.ci_upper),
        ("ci_half_width", results.ci_half_width),
    ):
        header.extend(f"{name}_{channel}" for channel in CHANNELS)
        arrays.append(arr)
    header.extend(f"joint_velocity_{channel}" for channel in JOINT_VELOCITY_CHANNELS)
    arrays.append(results.joint_velocity)
    header.extend(f"cartesian_velocity_{channel}" for channel in CARTESIAN_VELOCITY_CHANNELS)
    arrays.append(results.cartesian_velocity)
    header.append("cartesian_speed")
    arrays.append(results.cartesian_speed[:, None])
    path = output_dir / f"{results.estimator_mode}_external_force_with_ci.csv"
    np.savetxt(path, np.column_stack(arrays), delimiter=",", header=",".join(header), comments="")
    return path


def plot_force_ci(results, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
    for axis, ax in enumerate(axes):
        ax.fill_between(
            results.time,
            results.ci_lower[:, axis],
            results.ci_upper[:, axis],
            color="tab:red",
            alpha=0.20,
            linewidth=0,
            label="95% interval",
        )
        ax.plot(results.time, results.real_force[:, axis], color="tab:blue", linewidth=1.0, label="measured")
        ax.plot(results.time, results.force[:, axis], color="tab:red", linewidth=1.0, label=results.estimator_mode)
        ax.set_title(f"{CHANNELS[axis]}, validation RMSE = {results.rmse[axis]:.4f}")
        ax.set_ylabel("Force (N)")
        ax.grid(True, alpha=0.35)
    axes[-1].set_xlabel("Time (s)")
    axes[0].legend(loc="best")
    path = output_dir / f"{results.estimator_mode}_external_force_ci.png"
    fig.savefig(path, dpi=180)
    plt.show()
    return path


def plot_sigma_model(results, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 4))
    z = 1.959963984540054
    for channel in range(3):
        ax.plot(
            results.sigma_bin_centers,
            z * results.sigma_by_channel[:, channel],
            marker="o",
            linestyle="none",
            label=f"{CHANNELS[channel]} half-width model",
        )
    ax.set_xlabel("Estimated contact-force magnitude |F_hat| (N)")
    ax.set_ylabel("Median 95% half-width (N)")
    ax.set_title("Leak-free CI width versus estimated force magnitude")
    ax.grid(True, alpha=0.35)
    ax.legend(loc="best")
    fig.tight_layout()
    path = output_dir / f"{results.estimator_mode}_ci_width_vs_contact_force.png"
    fig.savefig(path, dpi=180)
    plt.show()
    return path


def centered_moving_average(data, window_size):
    if window_size <= 1:
        return data
    pad_before = window_size // 2
    pad_after = window_size - 1 - pad_before
    padded = np.pad(data, [(pad_before, pad_after)] + [(0, 0)] * (data.ndim - 1), mode="edge")
    kernel = np.ones(window_size, dtype=float) / float(window_size)
    return np.apply_along_axis(lambda col: np.convolve(col, kernel, mode="valid"), 0, padded)


def plot_velocity_uncertainty_context(results, output_dir, smoothing_window=101):
    output_dir.mkdir(parents=True, exist_ok=True)
    speed = centered_moving_average(results.cartesian_speed[:, None], smoothing_window)[:, 0]
    half_width = centered_moving_average(results.ci_half_width[:, :3], smoothing_window)
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, constrained_layout=True)
    axes[0].plot(results.time, speed, color="0.15", linewidth=1.2)
    axes[0].set_ylabel("Tool speed")
    axes[0].set_title("Velocity context for force uncertainty")
    axes[0].grid(True, alpha=0.35)
    for channel in range(3):
        axes[1].plot(results.time, half_width[:, channel], linewidth=1.2, label=f"{CHANNELS[channel]} CI half-width")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Smoothed force CI half-width (N)")
    axes[1].grid(True, alpha=0.35)
    axes[1].legend(loc="best")
    path = output_dir / f"{results.estimator_mode}_velocity_with_force_uncertainty.png"
    fig.savefig(path, dpi=180)
    plt.show()
    return path


active_csv = save_generalization_results_csv(active_results, OUTPUT_DIR)
force_ci_png = plot_force_ci(active_results, OUTPUT_DIR)
ci_width_png = plot_sigma_model(active_results, OUTPUT_DIR)
velocity_context_png = plot_velocity_uncertainty_context(active_results, OUTPUT_DIR)

print("Saved:")
print(active_csv)
print(force_ci_png)
print(ci_width_png)
print(velocity_context_png)

### Generalization readout

The direct-compute baseline and all observer/Kalman variants now start from the same torque residual:

$$
	au_{\mathrm{meas}} - 	au_{\mathrm{LSTM}}.
$$

So this section is an apples-to-apples comparison of filtering/CI methods on the exported torque path, without using `lstm_force.csv` as a shortcut.

### How to switch variants

Change only this line in the variant cell above:

```python
ACTIVE_VARIANT = "kalman_smooth_deadband"
```

Available options are:

```text
direct
observer
kalman
torque_smooth_deadband
kalman_smooth_deadband
bias_gate
two_time_scale
```

The hyperparameters are intentionally shared with the previous dataset so this notebook tests generalization rather than retuning.